In [1]:
# handle the display
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [2]:
# imports
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [3]:
from pyspark import SparkConf,SparkContext

conf = SparkConf().setMaster("local").setAppName("Word-Count-example")
sc = SparkContext(conf= conf)


24/01/09 15:42:55 WARN Utils: Your hostname, nalain-dell resolves to a loopback address: 127.0.1.1; using 10.200.120.6 instead (on interface wlp0s20f3)
24/01/09 15:42:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/01/09 15:42:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/01/09 15:42:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
path_to_file = Path("data/millionsfromwaste.txt")

book = sc.textFile(str(path_to_file))
book.collect()[:10]

['The Project Gutenberg eBook of Millions from Waste',
 '    ',
 'This ebook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this ebook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '']

In [9]:
word_counts = book.flatMap(lambda x: x.split()).countByValue()

for i, (word,count) in enumerate(word_counts.items()):
    print(f"{i} - {word} :  {count} ")
    if i == 10:
        break
    

0 - The :  995 
1 - Project :  79 
2 - Gutenberg :  22 
3 - eBook :  4 
4 - of :  4980 
5 - Millions :  3 
6 - from :  923 
7 - Waste :  15 
8 - This :  185 
9 - ebook :  2 
10 - is :  1828 


## improve a little

remove punctuations 

In [11]:
import re 

def preprocess_word(word):
    return re.sub("A-Za-z0-9]+","",word.lower())
def preprocess_words(words):
    return [preprocess_word(word) for word in words.split()]

In [12]:
preprocess_words('The Project Gutenberg eBook of Millions from Waste')

['the', 'project', 'gutenberg', 'ebook', 'of', 'millions', 'from', 'waste']

In [13]:
word_counts = book.flatMap(preprocess_words).countByValue()

for i, (word,count) in enumerate(word_counts.items()):
    print(f"{i} - {word} :  {count} ")
    if i == 10:
        break

0 - the :  10097 
1 - project :  85 
2 - gutenberg :  25 
3 - ebook :  8 
4 - of :  5029 
5 - millions :  21 
6 - from :  973 
7 - waste :  338 
8 - this :  1098 
9 - is :  1829 
10 - for :  1093 


In [15]:
# another way

words = book.flatMap(preprocess_words)
words_count = words.map(lambda x: (x,1)).reduceByKey(lambda x,y : x+y)
words_count.collect()[:10]

[('the', 10097),
 ('project', 85),
 ('gutenberg', 25),
 ('ebook', 8),
 ('of', 5029),
 ('millions', 21),
 ('from', 973),
 ('waste', 338),
 ('this', 1098),
 ('is', 1829)]

In [23]:
## problem is stop words

word_count_sorted = words_count.map(lambda x: (x[1],x[0])).sortByKey(False).map(lambda x: (x[1],x[0]))
word_count_sorted.collect()[:10]

[('the', 10097),
 ('of', 5029),
 ('to', 4289),
 ('and', 2452),
 ('in', 2215),
 ('a', 2080),
 ('is', 1829),
 ('be', 1394),
 ('it', 1328),
 ('as', 1201)]

## better preprocessing

use nltk

- remove stop words ....

In [26]:
## use nltk to remove stop words and advanced preprocessing 

!pip install nltk -q

In [28]:
import nltk 

nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words= stopwords.words("english")
stop_words[:10]


[nltk_data] Downloading package stopwords to /home/nalain/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're"]

In [34]:
# modified proprecessing

import re 

def preprocess_word(word):
    return re.sub("A-Za-z0-9]+"," ",word.lower())
def preprocess_words(words):
    preprocessed_words= [preprocess_word(word) for word in words.split()]
    return [word for word in preprocessed_words if word not in stop_words and word!=""]

In [35]:
preprocess_words('The Project Gutenberg eBook of Millions from Waste')

['project', 'gutenberg', 'ebook', 'millions', 'waste']

In [36]:
# another way

words = book.flatMap(preprocess_words)
words_count = words.map(lambda x: (x,1)).reduceByKey(lambda x,y : x+y)
word_count_sorted = words_count.map(lambda x: (x[1],x[0])).sortByKey(False).map(lambda x: (x[1],x[0]))
word_count_sorted.collect()[:10]

[('waste', 338),
 ('upon', 337),
 ('one', 335),
 ('would', 302),
 ('may', 285),
 ('material', 208),
 ('per', 198),
 ('every', 161),
 ('tons', 160),
 ('|', 159)]

In [38]:
sc.stop()